# Шаг 8. Доуточнение русских результатов и тексты длиннее 256 токенов

1. **Украинские тексты в русской части M4?** Первый текст в шаге 7 оказался на украинском. Считаем, сколько таких текстов, и как их исключение влияет на точность.
2. **Э2 для детекторов на признаках** — пересчёт с исправленной (правой) обрезкой русских текстов.
3. **Квантование ruGPT-3** — все варианты заново с правой обрезкой (для таблицы квантования).
4. **512 токенов вместо 256:** точность (Э4, обучение заново для каждой длины) и время на текст.

Время: ≈40 минут (в основном пункт 4). **Перед запуском:** T4 GPU, «Выполнить все».
**Прислать:** вывод ячеек 3–7 и файл `results_step8.json`.

## 1. Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, torch
WORK = '/content/drive/MyDrive/ai-text-detector'
for d in ['/content/code', '/content/m']: os.makedirs(d, exist_ok=True)
os.chdir('/content/code'); sys.path.insert(0, '/content/code')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'Нужна среда с GPU: Среда выполнения → Сменить среду выполнения → T4 GPU'
!pip -q install onnx onnxruntime 2>&1 | tail -1

## 2. Код проекта

In [ ]:
%%writefile /content/code/stylometry.py
"""Стилометрические признаки (та же логика будет реализована в расширении на JavaScript)."""
import re, math, unicodedata
from collections import Counter

RU_FUNC = set("и в во не на я что он с со как а то все она так его но да ты к ко у же вы за бы по только ее её мне было вот от меня еще ещё нет о об из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между".split())
EN_FUNC = set("the of and to a in is that it for on as with was be by this are or from at an but not have has had which they you he she we his her their its can will would there been were if all so no one more also do than about into such these other only some may then".split())

CYR2LAT = {"А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","Х":"X",
           "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x"}
LAT2CYR = {v: k for k, v in CYR2LAT.items()}

def cyr_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('Ѐ' <= ch <= 'ӿ' for ch in letters) / len(letters)

def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('a' <= ch.lower() <= 'z' for ch in letters) / len(letters)

def detect_lang(text):
    """'ru' — преобладает кириллица, 'en' — почти только латиница, иначе 'other' (не поддерживается)."""
    if cyr_ratio(text) > 0.5: return 'ru'
    if latin_ratio(text) > 0.9: return 'en'
    return 'other'

def _convert(word, table):
    return ''.join(table.get(ch, ch) for ch in word)

def _mixed(word):
    return any('\u0400' <= ch <= '\u04FF' for ch in word) and any('a' <= ch.lower() <= 'z' for ch in word)

def _fix_word(word, doc_script):
    """Слово из смеси алфавитов приводим к основному алфавиту текста; если это невозможно
    (в слове есть буквы без двойников) — к алфавиту большинства букв слова (защита от гомоглифов)."""
    if not _mixed(word):
        # слово целиком из «чужого» алфавита, но только из букв-двойников (например, кириллическая «а» в английском тексте)
        foreign = CYR2LAT if doc_script == 'lat' else LAT2CYR
        if word and all(ch in foreign for ch in word):
            return _convert(word, foreign)
        return word
    first = _convert(word, LAT2CYR if doc_script == 'cyr' else CYR2LAT)
    if not _mixed(first):
        return first
    cyr = sum('\u0400' <= ch <= '\u04FF' for ch in word)
    lat = sum('a' <= ch.lower() <= 'z' for ch in word)
    return _convert(word, LAT2CYR if cyr > lat else CYR2LAT)

def normalize(text):
    """NFKC, удаление невидимых символов, нормализация пробелов, исправление гомоглифов."""
    t = unicodedata.normalize('NFKC', text)
    t = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', t)
    t = re.sub(r'[ \t\u00a0]+', ' ', t)
    t = re.sub(r'\s*\n\s*', '\n', t).strip()
    doc_script = 'cyr' if cyr_ratio(t) > 0.5 else 'lat'
    return re.sub(r'\w+', lambda m: _fix_word(m.group(0), doc_script), t)

SENT_SPLIT = re.compile(r'(?<=[.!?…])\s+')
WORD = re.compile(r"[^\W\d_]+(?:[-'’][^\W\d_]+)*", re.UNICODE)

FEATURE_NAMES = ['mean_sent_len', 'cv_sent_len', 'mean_word_len', 'ttr', 'punct_ratio',
                 'func_ratio', 'rep_bigram_ratio', 'rep_trigram_ratio', 'comma_per_sent']

def stylometric_features(text, lang=None):
    lang = lang or detect_lang(text)
    sents = [s for s in SENT_SPLIT.split(text) if s.strip()]
    words = [w.lower() for w in WORD.findall(text)]
    n = max(len(words), 1)
    slens = [len(WORD.findall(s)) for s in sents] or [0]
    mu = sum(slens) / len(slens)
    sd = math.sqrt(sum((x - mu) ** 2 for x in slens) / len(slens))
    func = RU_FUNC if lang == 'ru' else EN_FUNC
    def rep(k):
        grams = [tuple(words[i:i+k]) for i in range(len(words) - k + 1)]
        if not grams: return 0.0
        c = Counter(grams); return sum(v for v in c.values() if v > 1) / len(grams)
    chars = max(len(text), 1)
    return {
        'mean_sent_len': mu,
        'cv_sent_len': sd / mu if mu else 0.0,
        'mean_word_len': sum(map(len, words)) / n,
        'ttr': len(set(words)) / n,
        'punct_ratio': sum(unicodedata.category(ch).startswith('P') for ch in text) / chars,
        'func_ratio': sum(w in func for w in words) / n,
        'rep_bigram_ratio': rep(2),
        'rep_trigram_ratio': rep(3),
        'comma_per_sent': text.count(',') / max(len(sents), 1),
    }

In [ ]:
%%writefile /content/code/metrics_utils.py
"""Метрики качества детектора (раздел 1.6)."""
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, precision_score, recall_score

def tpr_at_fpr(y, p, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, p)
    ok = fpr <= fpr_target
    return float(tpr[ok].max()) if ok.any() else 0.0

def evaluate(y, p, threshold=0.5, n_boot=200, seed=0):
    y, p = np.asarray(y), np.asarray(p); yhat = (p >= threshold).astype(int)
    res = {'n': int(len(y)), 'F1': f1_score(y, yhat), 'Precision': precision_score(y, yhat, zero_division=0),
           'Recall': recall_score(y, yhat), 'FPR': float(((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)),
           'AUROC': roc_auc_score(y, p), 'TPR@1%FPR': tpr_at_fpr(y, p)}
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):                       # доверительный интервал AUROC (бутстреп)
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) == 2: boots.append(roc_auc_score(y[idx], p[idx]))
    res['AUROC_95CI'] = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else None
    return res

In [ ]:
%%writefile /content/code/baseline_stylometry.py
"""Базовый стилометрический детектор: признаки + логистическая регрессия, отдельно для каждого языка."""
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from stylometry import stylometric_features, normalize, FEATURE_NAMES
from metrics_utils import evaluate

def _features_with_format(text, lang):
    t = normalize(text)
    f = stylometric_features(t, lang)
    f['newline_ratio'] = t.count('\n') / max(len(t), 1)   # признак форматирования — только для эксперимента-абляции
    return f

def featurize(df, names=FEATURE_NAMES):
    return np.array([[f[k] for k in names] for f in
                     (_features_with_format(t, l) for t, l in zip(df.text, df.lang))])

def split(df, seed=42):
    """70/15/15 с разбиением по pair_id: человеческий текст и его сгенерированная пара не разделяются."""
    g1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr, rest = next(g1.split(df, groups=df.pair_id))
    rest_df = df.iloc[rest]
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    va, te = next(g2.split(rest_df, groups=rest_df.pair_id))
    return df.iloc[tr], rest_df.iloc[va], rest_df.iloc[te]

def run(df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    d = df[df.lang == lang]
    d = pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])
    if d.label.nunique() < 2:
        return None, None, None
    tr, va, te = split(d, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'val': len(va), 'test': len(te)})
    coefs = pd.Series(model[-1].coef_[0], index=names).sort_values(key=abs, ascending=False)
    return res, coefs, model

def export_weights(models, path):
    """Сохраняет параметры нормировки и логистической регрессии в JSON для расширения (extension/model/weights.json)."""
    import json
    out = {'version': 'stylometry-v1', 'demo': False, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        out['langs'][lang] = {'features': FEATURE_NAMES, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=1)

def balanced_sample(d, cap_per_class, seed=42):
    return pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])

def run_cross(train_df, test_df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    """Обучение на одном корпусе и проверка на другом (перенос на незнакомые генераторы, серия Э3)."""
    tr = balanced_sample(train_df[train_df.lang == lang], cap_per_class, seed)
    te = balanced_sample(test_df[test_df.lang == lang], cap_per_class // 4, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'test': len(te)})
    return res

In [ ]:
%%writefile /content/code/lm_features.py
"""Вероятностные признаки по компактной авторегрессионной языковой модели (раздел 2.3, формулы 2.1–2.2).
Для каждого токена считаются: логарифм вероятности, энтропия предсказанного распределения, ранг
фактического токена и дисперсия логарифма вероятности (для признака условной кривизны, как в Fast-DetectGPT)."""
import numpy as np

LM_FEATURE_NAMES = ['mean_lp', 'std_lp', 'mean_entropy', 'mean_log_rank',
                    'rank_1', 'rank_2_10', 'rank_11_100', 'rank_101_1000', 'rank_gt_1000', 'curvature']

def token_stats(texts, model, tok, max_tokens=256, batch_size=8, device='cuda'):
    """Один проход модели по каждому тексту. Возвращает список словарей с массивами по токенам."""
    import torch
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    out = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors='pt', truncation=True, max_length=max_tokens,
                  padding=True, add_special_tokens=False).to(device)
        with torch.no_grad():
            logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits.float()
        logp = torch.log_softmax(logits[:, :-1], dim=-1)          # прогноз для токенов 2..T
        target = enc.input_ids[:, 1:]
        lp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)    # log P(w_i | w_<i)
        p = logp.exp()
        ent = -(p * logp).sum(-1)                                  # энтропия, формула (2.2)
        var = (p * logp.pow(2)).sum(-1) - ent.pow(2)               # дисперсия log p по распределению модели
        rank = (logp > lp.unsqueeze(-1)).sum(-1) + 1               # ранг фактического токена
        mask = enc.attention_mask[:, 1:].bool()
        for b in range(len(batch)):
            m = mask[b]
            out.append({'ids': enc.input_ids[b][enc.attention_mask[b].bool()].cpu().numpy(),
                        'lp': lp[b][m].cpu().numpy(), 'ent': ent[b][m].cpu().numpy(),
                        'var': var[b][m].cpu().numpy(), 'rank': rank[b][m].cpu().numpy()})
        del logits, logp, p
    return out

def aggregate(s, k=None):
    """Признаки по первым k предсказанным токенам (k=None — все)."""
    lp, ent, var, rank = (s[x][:k] if k else s[x] for x in ('lp', 'ent', 'var', 'rank'))
    n = len(lp)
    if n == 0:
        return {f: 0.0 for f in LM_FEATURE_NAMES}
    curv = (lp.sum() + ent.sum()) / np.sqrt(max(var.sum(), 1e-8))   # (∑log p − ∑E[log p]) / √∑Var, E[log p] = −H
    return {
        'mean_lp': float(lp.mean()), 'std_lp': float(lp.std()), 'mean_entropy': float(ent.mean()),
        'mean_log_rank': float(np.log(rank).mean()),
        'rank_1': float((rank == 1).mean()), 'rank_2_10': float(((rank >= 2) & (rank <= 10)).mean()),
        'rank_11_100': float(((rank >= 11) & (rank <= 100)).mean()),
        'rank_101_1000': float(((rank >= 101) & (rank <= 1000)).mean()), 'rank_gt_1000': float((rank > 1000).mean()),
        'curvature': float(curv),
    }


# Дополнительные признаки (шаг 6): вычисляются из тех же величин по токенам, без новых проходов модели.
LM_EXTRA_NAMES = ['lp_q10', 'lp_q50', 'lp_q90', 'std_entropy', 'mean_lp_plus_h',
                  'xent_ratio', 'frac_above_expected', 'mean_var']

def aggregate_ext(s, k=None):
    """lp_q10/50/90 — квантили логарифма вероятности; std_entropy — разброс энтропии;
    mean_lp_plus_h — среднее (log p + H), то есть превышение правдоподобия над ожидаемым в каждой позиции;
    xent_ratio — отношение среднего «удивления» −log p к средней энтропии (одномодельный аналог Binoculars);
    frac_above_expected — доля токенов, для которых log p > −H; mean_var — средняя дисперсия log p."""
    lp, ent, var = (np.asarray(s[x][:k] if k else s[x], dtype=np.float64) for x in ('lp', 'ent', 'var'))
    if len(lp) == 0:
        return {f: 0.0 for f in LM_EXTRA_NAMES}
    return {
        'lp_q10': float(np.quantile(lp, 0.1)), 'lp_q50': float(np.quantile(lp, 0.5)), 'lp_q90': float(np.quantile(lp, 0.9)),
        'std_entropy': float(ent.std()), 'mean_lp_plus_h': float((lp + ent).mean()),
        'xent_ratio': float((-lp).mean() / max(ent.mean(), 1e-6)),
        'frac_above_expected': float((lp > -ent).mean()), 'mean_var': float(var.mean()),
    }

In [ ]:
%%writefile /content/code/hybrid.py
"""Обучение и оценка статистического, стилометрического и гибридного детекторов (разделы 2.3, 3.3)."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from stylometry import stylometric_features, normalize, FEATURE_NAMES as STYLO
from lm_features import aggregate, LM_FEATURE_NAMES as LM
from metrics_utils import evaluate

VARIANTS = {'stylometric': STYLO, 'statistical': LM, 'hybrid': STYLO + LM}

def feature_table(texts, langs, stats, k=None, decode=None):
    """Признаки обеих групп. Если задан k, стилометрия считается по тексту первых k+1 токенов."""
    rows = []
    for t, l, s in zip(texts, langs, stats):
        if k is not None and decode is not None:
            t = decode(s['ids'][:k + 1], l)
        f = stylometric_features(t, l)
        f.update(aggregate(s, k))
        rows.append(f)
    return pd.DataFrame(rows)

def fit(X, y, sample_weight=None):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
    if sample_weight is None:
        return m.fit(X, y)
    return m.fit(X, y, logisticregression__sample_weight=sample_weight)

def threshold_at_fpr(model, X_val, y_val, fpr):
    """Порог, при котором на валидационной выборке доля ложных срабатываний равна fpr."""
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))

def train_eval_all(F_tr, y_tr, F_te, y_te, variants=VARIANTS):
    out, models = {}, {}
    for name, cols in variants.items():
        m = fit(F_tr[cols].values, y_tr)
        out[name] = evaluate(y_te, m.predict_proba(F_te[cols].values)[:, 1])
        models[name] = m
    return out, models

def export(models, thresholds, path, meta):
    """Веса гибридного детектора для расширения."""
    res = {'version': 'hybrid-v1', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        res['langs'][lang] = {'features': STYLO + LM, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]),
                              'thresholds': thresholds[lang]}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1)

In [ ]:
%%writefile /content/code/hybrid_v2.py
"""Гибридный детектор, версия 2 (шаг 6): дополнительные вероятностные признаки, подбор регуляризации
на валидационной выборке, обучение на нескольких корпусах. Классификатор — по-прежнему логистическая регрессия,
поэтому результат объясним, а в расширении он вычисляется теми же несколькими арифметическими операциями."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from stylometry import stylometric_features, FEATURE_NAMES as STYLO
from lm_features import aggregate, aggregate_ext, LM_FEATURE_NAMES as LM, LM_EXTRA_NAMES as EXTRA

SETS = {'stylometric': STYLO, 'hybrid_v1': STYLO + LM, 'hybrid_v2': STYLO + LM + EXTRA}
CS = (0.01, 0.03, 0.1, 0.3, 1.0, 3.0)


def feature_table(texts, langs, stats):
    rows = []
    for t, l, s in zip(texts, langs, stats):
        f = stylometric_features(t, l); f.update(aggregate(s)); f.update(aggregate_ext(s)); rows.append(f)
    return pd.DataFrame(rows)


def fit(X, y, C=1.0, sample_weight=None):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight='balanced'))
    if sample_weight is None:
        return m.fit(X, y)
    return m.fit(X, y, logisticregression__sample_weight=sample_weight)


def select_c(Xtr, ytr, val_sets, wtr=None, Cs=CS):
    """Подбор коэффициента регуляризации C по среднему AUROC на валидационных выборках val_sets = [(X, y), ...]."""
    best = None
    for C in Cs:
        m = fit(Xtr, ytr, C, wtr)
        score = float(np.mean([roc_auc_score(y, m.predict_proba(X)[:, 1]) for X, y in val_sets]))
        if best is None or score > best[1]:
            best = (C, score, m)
    return best   # (C, средний AUROC на валидации, модель)


def corpus_weights(groups):
    """Веса примеров: суммарный вес каждого корпуса одинаков."""
    groups = np.asarray(groups); w = np.ones(len(groups))
    for g in np.unique(groups):
        w[groups == g] = len(groups) / (len(np.unique(groups)) * (groups == g).sum())
    return w


def threshold_at_fpr(model, X_val, y_val, fpr):
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))


def model_json(model, cols, thresholds, C):
    sc, lr = model[0], model[-1]
    return {'features': list(cols), 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
            'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]), 'thresholds': thresholds, 'C': C}


def export(langs, path, meta):
    """langs = {lang: {'main': dict, 'stage1': dict}} — основной классификатор и быстрый стилометрический (первая ступень)."""
    res = {'version': 'hybrid-v2', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, d in langs.items():
        res['langs'][lang] = dict(d['main']); res['langs'][lang]['stage1'] = d['stage1']
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1, ensure_ascii=False)

In [ ]:
%%writefile /content/code/transfer_hybrid.py
"""Межъязыковой перенос детекторов на признаках (серия Э2): стилометрический, статистический и гибридный.
Английский и русский детекторы используют разные языковые модели (GPT-2 и ruGPT-3), но одинаковый набор признаков,
поэтому классификатор, обученный на английских признаках, можно применить к русским."""
import numpy as np
from hybrid import fit, VARIANTS
from metrics_utils import evaluate


def moments(F):
    """Среднее и стандартное отклонение каждого признака (метки не нужны)."""
    mu = F.mean(0); sd = F.std(0); sd[sd == 0] = 1.0
    return mu, sd


def align(F_target, src_moments, tgt_moments):
    """Нормировка по языку: распределение каждого признака целевого языка приводится к среднему и разбросу
    исходного языка. Используются только неразмеченные тексты целевого языка."""
    (mu_s, sd_s), (mu_t, sd_t) = src_moments, tgt_moments
    return (F_target - mu_t) / sd_t * sd_s + mu_s


def zero_shot(F_en_tr, y_en_tr, F_ru_tr, F_ru_te, y_ru_te, variants=VARIANTS, n_boot=200):
    """Обучение только на английских текстах, проверка на русских: без адаптации и с нормировкой по языку."""
    out, models = {}, {}
    for name, cols in variants.items():
        m = fit(F_en_tr[cols].values, y_en_tr); models[name] = m
        Xte = F_ru_te[cols].values
        out[name] = {'без адаптации': evaluate(y_ru_te, m.predict_proba(Xte)[:, 1], n_boot=n_boot)}
        Xal = align(Xte, moments(F_en_tr[cols].values), moments(F_ru_tr[cols].values))
        out[name]['нормировка по языку'] = evaluate(y_ru_te, m.predict_proba(Xal)[:, 1], n_boot=n_boot)
    return out, models


def few_shot(F_en_tr, y_en_tr, F_ru_tr, y_ru_tr, F_ru_te, y_ru_te, idx, variants=VARIANTS):
    """idx — индексы n русских обучающих примеров. Варианты:
    'только русские' — классификатор обучается на n русских примерах;
    'англ. + русские' — на всех английских и n русских примерах, русские признаки нормированы по языку
    (по всем неразмеченным русским обучающим текстам), суммарный вес русских примеров равен весу английских."""
    out = {}
    y_ru_tr = np.asarray(y_ru_tr); y_en_tr = np.asarray(y_en_tr)
    for name, cols in variants.items():
        Xr = F_ru_tr[cols].values; Xe = F_en_tr[cols].values; Xte = F_ru_te[cols].values
        r = {}
        m = fit(Xr[idx], y_ru_tr[idx])
        r['только русские'] = evaluate(y_ru_te, m.predict_proba(Xte)[:, 1], n_boot=0)
        mom_en, mom_ru = moments(Xe), moments(Xr)
        X = np.vstack([Xe, align(Xr[idx], mom_en, mom_ru)])
        y = np.concatenate([y_en_tr, y_ru_tr[idx]])
        w = np.concatenate([np.ones(len(Xe)), np.full(len(idx), len(Xe) / len(idx))])
        m2 = fit(X, y, sample_weight=w)
        r['англ. + русские'] = evaluate(y_ru_te, m2.predict_proba(align(Xte, mom_en, mom_ru))[:, 1], n_boot=0)
        out[name] = r
    return out

In [ ]:
%%writefile /content/code/encoder.py
"""Дообучение многоязычного энкодера (XLM-RoBERTa) для обнаружения сгенерированного текста (серия Э2).
Обычный цикл обучения PyTorch: так меньше зависимости от версий библиотек, чем с Trainer."""
import math, time, random
import numpy as np


def set_seed(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def encode(tok, texts, max_len=256):
    """Токенизация один раз заранее: список списков идентификаторов токенов (без дополнения)."""
    return tok(list(texts), truncation=True, max_length=max_len, add_special_tokens=True)['input_ids']


def _batch(ids_list, pad_id, device):
    import torch
    L = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(ids_list), L), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x, dtype=torch.long); mask[i, :len(x)] = 1
    return ids.to(device), mask.to(device)


def train(init, ids, labels, pad_id, steps=None, epochs=None, batch_size=16, lr=2e-5, warmup=0.1,
          weight_decay=0.01, seed=0, device='cuda', log_every=100):
    """Дообучает классификатор. init — имя модели на Hugging Face или путь к сохранённой модели.
    Длительность задаётся либо числом шагов (steps), либо числом эпох (epochs)."""
    import torch
    from transformers import AutoModelForSequenceClassification
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(init, num_labels=2).to(device)
    labels = np.asarray(labels, dtype=np.int64); n = len(labels)
    per_epoch = math.ceil(n / batch_size)
    total = steps if steps is not None else per_epoch * epochs
    no_decay = ('bias', 'LayerNorm.weight', 'layer_norm.weight')
    groups = [{'params': [p for nm, p in model.named_parameters() if not any(k in nm for k in no_decay)], 'weight_decay': weight_decay},
              {'params': [p for nm, p in model.named_parameters() if any(k in nm for k in no_decay)], 'weight_decay': 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr)
    n_warm = max(1, int(total * warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / n_warm if s < n_warm else max(0.0, (total - s) / max(1, total - n_warm)))
    use_amp = device == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    rng = np.random.default_rng(seed)
    model.train(); step = 0; t0 = time.time(); run_loss = []
    while step < total:
        order = rng.permutation(n)                      # новая перестановка на каждую эпоху
        for b in range(0, n, batch_size):
            if step >= total: break
            idx = order[b:b + batch_size]
            x, m = _batch([ids[i] for i in idx], pad_id, device)
            y = torch.tensor(labels[idx], device=device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=m).logits
            loss = torch.nn.functional.cross_entropy(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss.append(loss.item()); step += 1
            if log_every and step % log_every == 0:
                print(f'    шаг {step}/{total}  loss={np.mean(run_loss[-log_every:]):.3f}  {time.time() - t0:.0f} с')
    model.eval()
    return model


def predict(model, ids, pad_id, batch_size=64, device='cuda'):
    """Вероятность класса «сгенерирован» (метка 1) для каждого текста."""
    import torch
    out = []
    order = np.argsort([len(x) for x in ids])          # близкие по длине тексты в одном пакете — быстрее
    with torch.no_grad():
        for b in range(0, len(ids), batch_size):
            idx = order[b:b + batch_size]
            x, m = _batch([ids[i] for i in idx], pad_id, device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                logits = model(input_ids=x, attention_mask=m).logits.float()
            out.append((idx, torch.softmax(logits, -1)[:, 1].cpu().numpy()))
    p = np.zeros(len(ids), dtype=np.float64)
    for idx, pr in out: p[idx] = pr
    return p


def free():
    """Освобождает память видеокарты. Перед вызовом удалите модель: del model."""
    import torch, gc
    gc.collect(); torch.cuda.empty_cache()


def few_shot_sample(labels, n, seed):
    """Индексы n примеров (по n/2 каждого класса) из обучающей выборки."""
    rng = np.random.default_rng(seed); labels = np.asarray(labels)
    pos = rng.choice(np.where(labels == 1)[0], n // 2, replace=False)
    neg = rng.choice(np.where(labels == 0)[0], n - n // 2, replace=False)
    return rng.permutation(np.concatenate([pos, neg]))

In [ ]:
%%writefile /content/code/onnx_utils.py
"""Экспорт языковых моделей в ONNX, квантование INT8 и вычисление признаков через ONNX Runtime (разделы 1.5, 4.1)."""
import os, time, numpy as np

def export_onnx(model, tok, out_dir, opset=17):
    """Экспорт модели (вход: input_ids, attention_mask; выход: logits) и токенизатора в формат для Transformers.js."""
    import torch
    os.makedirs(os.path.join(out_dir, 'onnx'), exist_ok=True)
    model = model.float().cpu().eval()
    model.config.use_cache = False

    class Wrap(torch.nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, input_ids, attention_mask):
            return self.m(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits

    ids = torch.ones(1, 8, dtype=torch.long); mask = torch.ones(1, 8, dtype=torch.long)
    path = os.path.join(out_dir, 'onnx', 'model.onnx')
    kw = dict(input_names=['input_ids', 'attention_mask'], output_names=['logits'], opset_version=opset,
              dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'},
                            'logits': {0: 'batch', 1: 'seq'}})
    try:
        torch.onnx.export(Wrap(model), (ids, mask), path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(Wrap(model), (ids, mask), path, **kw)
    model.config.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)          # быстрый токенизатор сохраняет tokenizer.json
    return path

def last_matmul_name(path):
    """Имя последнего умножения матриц — выходной слой модели (проекция на словарь)."""
    import onnx
    m = onnx.load(path, load_external_data=False)
    return [n.name for n in m.graph.node if n.op_type in ('MatMul', 'Gemm')][-1]

QUANT_VARIANTS = {
    'q8': {},                                         # по умолчанию: один масштаб на всю матрицу весов
    'q8_per_channel': {'per_channel': True},          # свой масштаб для каждого столбца
    'q8_keep_head': {'keep_head': True},              # выходной слой остаётся в fp32
    'q8_pc_keep_head': {'per_channel': True, 'keep_head': True},
}

def quantize(path_fp32, variant='q8'):
    from onnxruntime.quantization import quantize_dynamic, QuantType
    opts = QUANT_VARIANTS[variant]
    out = path_fp32.replace('model.onnx', f'model_{variant}.onnx')
    kw = {'per_channel': opts.get('per_channel', False)}
    if opts.get('keep_head'):
        kw['nodes_to_exclude'] = [last_matmul_name(path_fp32)]
    quantize_dynamic(path_fp32, out, weight_type=QuantType.QInt8, **kw)
    return out

def size_mb(path):
    return os.path.getsize(path) / 2**20

def ort_token_stats(session, tok, texts, max_tokens=256):
    """То же, что lm_features.token_stats, но через ONNX Runtime на CPU — по одному тексту, как в браузере."""
    out, times = [], []
    for t in texts:
        ids = tok(t, add_special_tokens=False, truncation=True, max_length=max_tokens, return_tensors='np')['input_ids'].astype(np.int64)
        if ids.shape[1] < 2:
            out.append({'ids': ids[0], 'lp': np.array([]), 'ent': np.array([]), 'var': np.array([]), 'rank': np.array([])}); continue
        t0 = time.perf_counter()
        logits = session.run(['logits'], {'input_ids': ids, 'attention_mask': np.ones_like(ids)})[0][0, :-1].astype(np.float64)
        times.append(time.perf_counter() - t0)
        mx = logits.max(1, keepdims=True)
        logp = logits - (np.log(np.exp(logits - mx).sum(1, keepdims=True)) + mx)
        tgt = ids[0, 1:]
        lp = logp[np.arange(len(tgt)), tgt]
        p = np.exp(logp)
        ent = -(p * logp).sum(1)
        var = (p * logp ** 2).sum(1) - ent ** 2
        rank = (logp > lp[:, None]).sum(1) + 1
        out.append({'ids': ids[0], 'lp': lp, 'ent': ent, 'var': var, 'rank': rank})
    return out, times

In [ ]:
%%writefile /content/code/lm_graph.py
"""Вычисление величин по токенам внутри графа модели ONNX (шаг 6).
Раньше модель возвращала логиты — по 50 тысяч чисел на каждый токен, и признаки считались по ним на JavaScript.
Теперь граф сам вычисляет для каждого токена 4 числа: log p, энтропию H, дисперсию log p и ранг."""
import os, time, numpy as np


def make_feature_model(lm):
    import torch

    class FeatureHead(torch.nn.Module):
        def __init__(self, lm):
            super().__init__(); self.lm = lm

        def forward(self, input_ids, attention_mask):
            logits = self.lm(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits[:, :-1, :].float()
            target = input_ids[:, 1:].unsqueeze(-1)
            logp = torch.log_softmax(logits, dim=-1)
            lp = torch.gather(logp, -1, target).squeeze(-1)                      # log P(w_i | w_<i)
            p = torch.exp(logp)
            H = -(p * logp).sum(-1)                                              # энтропия
            var = (p * (logp + H.unsqueeze(-1)) ** 2).sum(-1)                    # дисперсия log p (устойчивая форма)
            rank = (logp > lp.unsqueeze(-1)).sum(-1).to(torch.float32) + 1.0     # ранг фактического токена
            return torch.stack([lp, H, var, rank], dim=-1)                      # [batch, T-1, 4]
    return FeatureHead(lm)


def export_feature_graph(lm, tok, out_dir, opset=17):
    """Экспорт графа «модель + величины по токенам» (вход: input_ids, attention_mask; выход: stats) и токенизатора."""
    import torch
    os.makedirs(os.path.join(out_dir, 'onnx'), exist_ok=True)
    lm = lm.float().cpu().eval(); lm.config.use_cache = False
    m = make_feature_model(lm).eval()
    ids = torch.ones(1, 8, dtype=torch.long); mask = torch.ones(1, 8, dtype=torch.long)
    path = os.path.join(out_dir, 'onnx', 'model.onnx')
    kw = dict(input_names=['input_ids', 'attention_mask'], output_names=['stats'], opset_version=opset,
              dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'},
                            'stats': {0: 'batch', 1: 'seq_minus_1'}})
    with torch.no_grad():
        try:
            torch.onnx.export(m, (ids, mask), path, dynamo=False, **kw)
        except TypeError:
            torch.onnx.export(m, (ids, mask), path, **kw)
    lm.config.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return path


def graph_token_stats(session, tok, texts, max_tokens=256):
    """Величины по токенам из графа (по одному тексту, как в браузере). Время — полное: модель + разбор ответа."""
    out, times = [], []
    for t in texts:
        ids = tok(t, add_special_tokens=False, truncation=True, max_length=max_tokens, return_tensors='np')['input_ids'].astype(np.int64)
        if ids.shape[1] < 2:
            out.append({'ids': ids[0], 'lp': np.array([]), 'ent': np.array([]), 'var': np.array([]), 'rank': np.array([])}); continue
        t0 = time.perf_counter()
        st = session.run(['stats'], {'input_ids': ids, 'attention_mask': np.ones_like(ids)})[0][0].astype(np.float64)
        rec = {'ids': ids[0], 'lp': st[:, 0], 'ent': st[:, 1], 'var': st[:, 2], 'rank': st[:, 3].round().astype(np.int64)}
        times.append(time.perf_counter() - t0)
        out.append(rec)
    return out, times


def logits_token_stats_timed(session, tok, texts, max_tokens=256):
    """Прежний способ (граф отдаёт логиты, величины считаются снаружи) — с полным временем на текст."""
    from onnx_utils import ort_token_stats
    out, times = [], []
    for t in texts:
        t0 = time.perf_counter()
        st, _ = ort_token_stats(session, tok, [t], max_tokens)
        times.append(time.perf_counter() - t0); out.append(st[0])
    return out, times

## 3. Данные и украинские тексты

In [ ]:
import importlib, pickle, time, json, re, numpy as np, pandas as pd
import stylometry, metrics_utils, baseline_stylometry, lm_features, hybrid, hybrid_v2, transfer_hybrid, encoder, onnx_utils, lm_graph
for m in (stylometry, metrics_utils, baseline_stylometry, lm_features, hybrid, hybrid_v2, transfer_hybrid, encoder, onnx_utils, lm_graph): importlib.reload(m)
from stylometry import normalize, stylometric_features
from baseline_stylometry import split
from lm_features import token_stats, aggregate, aggregate_ext
from metrics_utils import evaluate
from hybrid_v2 import SETS, fit
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM
SEED = 42
df = pd.read_parquet(f'{WORK}/dataset_step1.parquet'); df = df[df.dataset == 'M4']
def bal(d, cap): return pd.concat([g.sample(min(len(g), cap), random_state=SEED) for _, g in d.groupby('label')])
data = {'en': bal(df[df.lang == 'en'], 6000), 'ru': df[df.lang == 'ru']}
for k, d in data.items():
    d = d.copy(); d['norm'] = d.text.astype(str).map(normalize); data[k] = d.reset_index(drop=True)
LM_NAMES = {'en': 'gpt2', 'ru': 'ai-forever/rugpt3small_based_on_gpt2'}
TOK = {l: AutoTokenizer.from_pretrained(n) for l, n in LM_NAMES.items()}
for t in TOK.values(): t.truncation_side = 'right'
STATS = {'en': pickle.load(open(f'{WORK}/lm_stats_step2.pkl', 'rb'))['en'], 'ru': pickle.load(open(f'{WORK}/lm_stats_ru_right.pkl', 'rb'))['ru']}
assert all(len(STATS[l]) == len(data[l]) for l in STATS), 'Кэш не совпадает с данными — напиши мне'
SPL = {}
for l in ['en', 'ru']:
    tr, va, te = split(data[l], SEED); SPL[l] = {'tr': tr.index.values, 'va': va.index.values, 'te': te.index.values}
Y = {l: data[l].label.values.astype(int) for l in data}
F = {l: hybrid_v2.feature_table(data[l].norm, [l] * len(data[l]), STATS[l]) for l in ['en', 'ru']}
def XY(l, part, cols, mask=None):
    idx = SPL[l][part]
    if mask is not None: idx = idx[mask[idx]]
    return F[l].iloc[idx][cols].values, Y[l][idx]
RES = {}
def save(): json.dump(RES, open(f'{WORK}/results_step8.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
# Украинские буквы, которых нет в русском алфавите: і ї є ґ
ua = data['ru'].norm.str.count(r'[іїєґІЇЄҐ]') / data['ru'].norm.str.len().clip(lower=1)
is_ua = (ua > 0.005).values
RES['ukrainian'] = {'share': float(is_ua.mean()), 'count': int(is_ua.sum()),
                    'by_label': {int(k): int(v) for k, v in pd.Series(Y['ru'][is_ua]).value_counts().items()},
                    'share_in_test': float(is_ua[SPL['ru']['te']].mean())}
print('Тексты с украинскими буквами в русской части M4:', RES['ukrainian'])
for t in data['ru'].norm[is_ua].head(3): print('   пример:', t[:100])
cols = SETS['hybrid_v1']; keep = ~is_ua
m_all = fit(*XY('ru', 'tr', cols)); m_ru = fit(*XY('ru', 'tr', cols, keep))
RES['ukrainian']['auroc_test_all_trained_all'] = float(roc_auc_score(XY('ru', 'te', cols)[1], m_all.predict_proba(XY('ru', 'te', cols)[0])[:, 1]))
RES['ukrainian']['auroc_test_ru_only_trained_all'] = float(roc_auc_score(XY('ru', 'te', cols, keep)[1], m_all.predict_proba(XY('ru', 'te', cols, keep)[0])[:, 1]))
RES['ukrainian']['auroc_test_ru_only_trained_ru_only'] = float(roc_auc_score(XY('ru', 'te', cols, keep)[1], m_ru.predict_proba(XY('ru', 'te', cols, keep)[0])[:, 1]))
print({k: round(v, 3) for k, v in RES['ukrainian'].items() if k.startswith('auroc')}); save()

## 4. Э2 для детекторов на признаках — с правой обрезкой

In [ ]:
from transfer_hybrid import zero_shot, few_shot
from encoder import few_shot_sample
from hybrid import VARIANTS
Fe, ye = F['en'].iloc[SPL['en']['tr']], Y['en'][SPL['en']['tr']]
Fr, yr = F['ru'].iloc[SPL['ru']['tr']], Y['ru'][SPL['ru']['tr']]
Ft, yt = F['ru'].iloc[SPL['ru']['te']], Y['ru'][SPL['ru']['te']]
zs, _ = zero_shot(Fe, ye, Fr, Ft, yt, variants=VARIANTS)
RES['E2_features'] = {'zero_shot': zs, 'few_shot': {}}
print('Только английский → русский тест:')
for name, v in zs.items(): print(f'  {name:12s}', {mode: round(r['AUROC'], 3) for mode, r in v.items()})
for n in [100, 500, 1000, 'all']:
    rs = [few_shot(Fe, ye, Fr, yr, Ft, yt, np.arange(len(yr)) if n == 'all' else few_shot_sample(yr, n, s), variants=VARIANTS)
          for s in ([0] if n == 'all' else [0, 1, 2])]
    RES['E2_features']['few_shot'][str(n)] = rs
    print(n, {f'{name} | {mode}': f"{np.mean([r[name][mode]['AUROC'] for r in rs]):.3f}" + (f"±{np.std([r[name][mode]['AUROC'] for r in rs]):.3f}" if len(rs) > 1 else '')
              for name in VARIANTS for mode in ['только русские', 'англ. + русские']})
save()

## 5. Квантование ruGPT-3 — все варианты с правой обрезкой (≈8 минут)

In [ ]:
import onnxruntime as ort, shutil
from onnx_utils import quantize, size_mb, QUANT_VARIANTS
from lm_graph import export_feature_graph, graph_token_stats
HYB = {l: fit(*XY(l, 'tr', SETS['hybrid_v1'])) for l in ['en', 'ru']}
P = {}
for l in ['en', 'ru']:
    fp32 = export_feature_graph(AutoModelForCausalLM.from_pretrained(LM_NAMES[l]), TOK[l], f'/content/m/{l}')
    P[l] = {'fp32': fp32}
    for v in (QUANT_VARIANTS if l == 'ru' else ['q8_pc_keep_head']): P[l][v] = quantize(fp32, v)
def sess(path): so = ort.SessionOptions(); so.intra_op_num_threads = 1; return ort.InferenceSession(path, so, providers=['CPUExecutionProvider'])
RES['quant_ru'] = {}
d = data['ru']; te = d.index[SPL['ru']['te']]
sub = pd.concat([g.sample(150, random_state=SEED) for _, g in d.loc[te].groupby('label')])
for v, path in P['ru'].items():
    st, t = graph_token_stats(sess(path), TOK['ru'], sub.norm.tolist())
    Fq = pd.DataFrame([{**stylometric_features(x, 'ru'), **aggregate(s)} for x, s in zip(sub.norm, st)])
    r = evaluate(sub.label.values, HYB['ru'].predict_proba(Fq[SETS['hybrid_v1']].values)[:, 1], n_boot=0)
    r['size_mb'] = round(size_mb(path), 1); r['ms_median'] = float(np.median(t) * 1000); r['ms_p95'] = float(np.percentile(t, 95) * 1000)
    RES['quant_ru'][v] = r; save()
    print(f'ru {v:16s} {r["size_mb"]:6.1f} МБ AUROC={r["AUROC"]:.3f} F1={r["F1"]:.3f} TPR@1%={r["TPR@1%FPR"]:.3f} | медиана {r["ms_median"]:.0f} мс, 95% {r["ms_p95"]:.0f} мс')

## 6. 512 токенов: точность (≈25 минут на подсчёт признаков)

In [ ]:
cache = f'{WORK}/lm_stats_512.pkl'
S5 = pickle.load(open(cache, 'rb')) if os.path.exists(cache) else {}
for l in ['en', 'ru']:
    if l in S5: continue
    lm = AutoModelForCausalLM.from_pretrained(LM_NAMES[l]).to(DEVICE); t0 = time.time()
    S5[l] = token_stats(list(data[l].norm), lm, TOK[l], max_tokens=512, batch_size=4, device=DEVICE)
    print(f'{l}: {len(S5[l])} текстов за {time.time() - t0:.0f} с'); pickle.dump(S5, open(cache, 'wb')); del lm; torch.cuda.empty_cache()
RES['E4_512'] = {}
for l in ['en', 'ru']:
    n_tok = np.array([len(s['ids']) for s in S5[l]])
    RES['E4_512'][l] = {'share_longer_256': float((n_tok > 256).mean()), 'share_reaching_512': float((n_tok >= 512).mean())}
    for k in [256, 512]:
        Fk = hybrid.feature_table(data[l].norm, [l] * len(data[l]), S5[l], k=k - 1, decode=lambda ids, lang: TOK[lang].decode(ids))
        row = {}
        for v in ['stylometric', 'statistical', 'hybrid']:
            cols = VARIANTS[v]; m = fit(Fk.iloc[SPL[l]['tr']][cols].values, Y[l][SPL[l]['tr']])
            row[v] = evaluate(Y[l][SPL[l]['te']], m.predict_proba(Fk.iloc[SPL[l]['te']][cols].values)[:, 1])
        RES['E4_512'][l][k] = row; save()
        print(l, k, {v: f"{r['AUROC']:.3f} [{r['AUROC_95CI'][0]:.3f}; {r['AUROC_95CI'][1]:.3f}]" for v, r in row.items()})
    print(l, 'доля текстов длиннее 256 токенов:', round(RES['E4_512'][l]['share_longer_256'], 3))

## 7. 512 токенов: время на текст (ONNX Runtime, процессор, 1 поток, квантованные модели)

In [ ]:
RES['time_512'] = {}
for l, v in [('en', 'q8_pc_keep_head'), ('ru', 'q8')]:
    s = sess(P[l][v]); long = [t for t, st in zip(data[l].norm, S5[l]) if len(st['ids']) >= 512][:40]
    row = {}
    for k in [256, 512]:
        times = []
        for t in long:
            ids = TOK[l](t, add_special_tokens=False, truncation=True, max_length=k, return_tensors='np')['input_ids'].astype(np.int64)
            t0 = time.perf_counter(); s.run(['stats'], {'input_ids': ids, 'attention_mask': np.ones_like(ids)}); times.append(time.perf_counter() - t0)
        row[k] = float(np.median(times) * 1000)
    RES['time_512'][l] = row; save()
    print(l, {k: f'{ms:.0f} мс' for k, ms in row.items()}, f'— во сколько раз дольше: {row[512] / row[256]:.2f}')
print('Сохранено: results_step8.json')